# Other Normalizing Flow Models: Glow and FFJORD

> **Notebook goal:** study two different extensions of the flow idea. Glow adds learned invertible channel mixing and ActNorm; FFJORD treats a flow as a continuous differential equation and tracks density changes through the divergence of a vector field.

The shared goal remains exact (or controllably approximated) density evaluation and invertible generation. The parameterization of the transformation differs, as do the trade-offs in computation, memory, and flexibility.

**Contents**
1. Glow: ActNorm, invertible 1×1 convolution, and affine coupling
2. Concise PyTorch implementations of Glow components and round-trip checks
3. FFJORD / continuous normalizing flows (CNFs): the continuity equation
4. A discretized 2D CNF and log-density estimation
5. Model-family comparison and practical guidance


## 1. Glow: improved channel mixing

RealNVP often uses a fixed permutation between coupling layers. Glow replaces it with a learned invertible matrix $W$ (for images, this is channel mixing via an invertible 1×1 convolution). For a tensor $h\in\mathbb R^{C\times H\times W}$, the same matrix is applied at every spatial position:

$$\log\left|\det J\right|=H\,W\,\log\left|\det(W)\right|.$$

Here, $H$ and $W$ denote spatial height and width; $W$ inside the determinant is the channel-mixing matrix. To avoid repeatedly computing a costly determinant of an unconstrained matrix, Glow parameterizes the matrix using an LU decomposition. The determinant then depends only on the product of the diagonal entries of $U$. For clarity, the implementation below uses a direct matrix parameterization without LU. This is suitable for a small number of channels; larger models should use LU.

**ActNorm** is a channel-wise affine transformation with learned scale and shift. It is initialized on the first batch so activations have zero mean and unit variance; after initialization, its parameters are ordinary learned weights. This stabilizes training independently of minibatch size.

A Glow step typically consists of ActNorm → invertible 1×1 convolution → affine coupling. These operations normalize scale, mix channels, and then introduce conditional nonlinearity.


In [ ]:
import math
import torch
import torch.nn as nn

torch.manual_seed(5)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class ActNorm(nn.Module):
    """ActNorm for NCHW. Initializes once on the first batch."""
    def __init__(self, channels, eps=1e-6):
        super().__init__(); self.bias = nn.Parameter(torch.zeros(1,channels,1,1)); self.log_scale = nn.Parameter(torch.zeros(1,channels,1,1))
        self.register_buffer('initialized', torch.tensor(False)); self.eps=eps
    @torch.no_grad()
    def initialize(self, x):
        mean = x.mean(dim=(0,2,3), keepdim=True); std = x.std(dim=(0,2,3), keepdim=True).clamp_min(self.eps)
        self.bias.copy_(-mean); self.log_scale.copy_(-std.log()); self.initialized.fill_(True)
    def forward(self, x):
        if not bool(self.initialized): self.initialize(x)
        y = (x + self.bias) * self.log_scale.exp()
        ld = x.shape[2]*x.shape[3]*self.log_scale.sum()
        return y, ld.expand(x.shape[0])
    def inverse(self, y): return y * (-self.log_scale).exp() - self.bias

class Invertible1x1(nn.Module):
    """Mała wersja z bezpośrednimi wagami; determinant liczymy przez slogdet."""
    def __init__(self, channels):
        super().__init__()
        q, _ = torch.linalg.qr(torch.randn(channels,channels))
        self.weight = nn.Parameter(q)
    def forward(self, x):
        _, logabsdet = torch.linalg.slogdet(self.weight)
        y = torch.einsum('ij,njhw->nihw', self.weight, x)
        ld = x.shape[2]*x.shape[3]*logabsdet
        return y, ld.expand(x.shape[0])
    def inverse(self, y):
        w_inv = torch.linalg.inv(self.weight)
        return torch.einsum('ij,njhw->nihw', w_inv, y)

an = ActNorm(4).to(device); conv = Invertible1x1(4).to(device)
x = torch.randn(8,4,6,6,device=device)
y, ld1 = an(x); xr = an.inverse(y)
v, ld2 = conv(y); yr = conv.inverse(v)
print('ActNorm max round-trip error:', (x-xr).abs().max().item())
print('1x1 conv max round-trip error:', (y-yr).abs().max().item())
print('Batch logdet:', ld1[:2].detach(), ld2[:2].detach())

ActNorm max round-trip error: 2.384185791015625e-07
1x1 conv max round-trip error: 4.76837158203125e-07
Batch logdet: tensor([5.2871, 5.2871]) tensor([-5.3644e-06, -5.3644e-06])


## 2. Affine coupling and multiscale images in Glow

Glow coupling follows the RealNVP pattern: some channels stay unchanged while a network predicts shift and scale for the others. The 1×1 convolutions learn to mix channels, making fixed masks sufficient. For images, Glow uses a squeeze operation: each 2×2 spatial neighborhood becomes additional channels. Some channels can then be factored out as latents at several scales. This reduces the workload of later layers and creates a hierarchical latent representation.

**Cost:** the convolution log-determinant scales with $H\times W$. Without LU, each spatial location requires a channel-matrix determinant. More channels increase the cost; LU makes the log-determinant cheap, but the diagonal parameters must remain numerically stable.

The code above implements two important Glow components, not a complete image model. A full Glow requires coupling networks, squeeze/split operations, a multiscale prior, and pixel dequantization. Keep component demonstrations distinct from complete models.


## 3. FFJORD: a flow in continuous time

A discrete flow applies a finite sequence of maps. A continuous normalizing flow describes a trajectory $z(t)$ with an ordinary differential equation:
$$\frac{dz(t)}{dt}=f_\theta(z(t),t),\quad t\in[0,T].$$

Instead of explicitly computing the large Jacobian of the endpoint map, we use the density continuity equation:
$$\frac{d}{dt}\log p(z(t))=-\operatorname{tr}\left(\frac{\partial f_\theta}{\partial z}\right).$$

The trace of the vector-field derivative is its divergence. We integrate it along the trajectory to correct the log density. FFJORD (Free-form Jacobian of Reversible Dynamics) uses the Hutchinson trace estimator: for a noise vector $\epsilon$ with $\mathbb E[\epsilon\epsilon^T]=I$, $\operatorname{tr}(J)=\mathbb E[\epsilon^T J\epsilon]$. This estimates the trace using Jacobian-vector products without constructing the full matrix.

The ODE solver is part of the model: tolerances and step counts affect both cost and accuracy. A CNF has flexible dynamics but may require many function evaluations (NFE); backpropagation through a solver can consume substantial memory. FFJORD uses adjoint sensitivity to reduce memory use at the cost of additional computation.


In [2]:
# Small pedagogical 2D CNF: fixed-step Euler integration and exact trace.
# This is not a production solver; it illustrates the equations and correction sign.
class VectorField(nn.Module):
    def __init__(self, hidden=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(3,hidden),nn.Tanh(),nn.Linear(hidden,hidden),nn.Tanh(),nn.Linear(hidden,2))
    def forward(self,z,t):
        time = torch.full((z.shape[0],1),float(t),device=z.device,dtype=z.dtype)
        return self.net(torch.cat([z,time],dim=-1))

def divergence_exact(field, z, t):
    # D=2: explicit trace, with a z-gradient for each component.
    z = z.requires_grad_(True); dz = field(z,t); tr = 0.
    for d in range(z.shape[1]):
        grad = torch.autograd.grad(dz[:,d].sum(),z,create_graph=True,retain_graph=True)[0]
        tr = tr + grad[:,d]
    return dz, tr

def integrate_cnf(field,z0,steps=32):
    z=z0; delta_logp=torch.zeros(z.shape[0],device=z.device)
    dt=1.0/steps
    for i in range(steps):
        t=i/steps
        dz, div = divergence_exact(field,z,t)
        z = z + dt*dz
        # d log p(z(t))/dt = -div f
        delta_logp = delta_logp - dt*div
    return z, delta_logp

field=VectorField().to(device)
z0=torch.randn(16,2,device=device)
z1, correction=integrate_cnf(field,z0,steps=8)
print('State shape:', z1.shape, '| log p correction (first 4):', correction[:4].detach())

State shape: torch.Size([16, 2]) | log p correction (first 4): tensor([0.1199, 0.0664, 0.1621, 0.1572])


## 4. Full likelihood with a CNF

Suppose $z(0)=x$ is an observation and the solver evolves it to $z(T)$, where the base density is known. Integration gives:
$$\log p(x)=\log p(z(T))+\int_0^T \operatorname{div} f(z(t),t)\,dt.$$

In the code, `delta_logp = -integral(div)` is the change in trajectory log density from start to finish. Therefore, $\log p(x)=\log p(z(T))-\texttt{delta\_logp}$. The sign follows from the continuity equation and is an easy place to make a mistake.

In 2D we can compute the exact trace. In dimension $D$, this would require $D$ gradients per function evaluation. The Hutchinson estimator draws $\epsilon$, differentiates the scalar $f(z)^T\epsilon$ with respect to $z$, then takes its inner product with $\epsilon$. Estimator variance can make optimization harder; many implementations keep the same epsilon throughout one ODE solve.


In [3]:
def cnf_log_prob(field, x, steps=16):
    zT, delta = integrate_cnf(field, x, steps=steps)
    log_p_base = -0.5*(zT.square()+math.log(2*math.pi)).sum(-1)
    return log_p_base - delta

lp = cnf_log_prob(field,z0,steps=8)
print('CNF log_prob shape:',lp.shape,'| finite:',torch.isfinite(lp).all().item())

def divergence_hutchinson(field,z,t,eps=None):
    """Nieobciążony estymator śladu; eps może być Rademacher lub Gauss."""
    if eps is None: eps=torch.empty_like(z).bernoulli_(0.5).mul_(2).sub_(1)
    z=z.requires_grad_(True); fz=field(z,t)
    jte=torch.autograd.grad((fz*eps).sum(),z,create_graph=True)[0]
    return (jte*eps).sum(-1)

ztest=torch.randn(5,2,device=device)
print('Exact divergence:',divergence_exact(field,ztest,0.2)[1].detach())
print('Hutchinson div:',divergence_hutchinson(field,ztest,0.2).detach())

CNF log_prob shape: torch.Size([16]) | finite: True
Exact divergence: tensor([-0.1790, -0.1983, -0.0921, -0.0603, -0.0671])
Hutchinson div: tensor([-0.1188, -0.1334, -0.1060, -0.0527, -0.0262])


## 5. Comparing model families and choosing a flow

| Family | Main idea | Log-density change | Typical strengths | Cost and limitations |
|---|---|---|---|---|
| RealNVP | Affine coupling, masks, and permutations | Sum of log-scales | Simple implementation, exact likelihood | Requires many layers; mixing is limited |
| Glow | ActNorm + learned 1×1 conv + coupling | Scale terms and $H\,W\,\log\lvert\det(W)\rvert$ | Better channel mixing, multiscale images | Large models; costly memory and computation |
| FFJORD/CNF | Continuous ODE dynamics | Integral of divergence | Flexible trajectories and vector fields | Solver/NFE cost, gradient cost, numerical error |

Other important directions include planar/radial flows (simple variational flows), autoregressive flows (MAF/IAF, with different density-evaluation and sampling trade-offs), spline flows (more flexible monotone 1D transforms), and residual flows. The name “flow” covers many layer families, but the change-of-variables calculation remains the common foundation.

## Summary and next steps

Glow shows how to learn channel mixing, while ActNorm improves activation scaling. FFJORD replaces a discrete list of layers with a continuous trajectory and a Jacobian trace with an integral of divergence. The right choice depends on the data, likelihood requirements, compute budget, and whether fast sampling or density evaluation is the priority.

Suggested exercises: measure round-trip error and log-determinants in RealNVP; add ActNorm and 1×1 convolution; compare exact trace with Hutchinson in 2D; reduce the solver step size and observe stability. For full image models, use a proper multiscale architecture, dequantization, and a fixed benchmark for validation.

## Key takeaways

- Glow combines data-dependent ActNorm initialization, learned invertible channel mixing, and affine coupling.
- For an invertible 1×1 convolution, the log-determinant contribution is $H\,W\,\log\lvert\det(W)\rvert$, since the same channel transform is applied at every spatial location.
- A continuous normalizing flow follows an ODE, and its log density changes at the negative divergence of the vector field.
- FFJORD uses Hutchinson's estimator to estimate the Jacobian trace without constructing the full Jacobian.
- CNFs trade flexible dynamics for solver and gradient cost; track solver accuracy and the number of function evaluations.
